<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/15_comparacion_validacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 15. Comparación en validación

Junta todas las corridas del equipo guardadas en `Mi unidad/PDI-TP/resultados/` y las compara sobre las mismas
162 imágenes de val y sobre las 23 de Jamaica (el test va a ser de Kingston). **No entrena nada**: abre los
resultados en solo lectura (`entorno.preparar(solo_lectura=True)`), así que no puede crear ni modificar nada en
Drive.

Entran en la comparación:

- **Clasificadores**: MobileNetV2, ResNet50 y EfficientNetB0 en sus tres modos, **solo las corridas con la receta
  común vigente** (`receta: "comun_v2"` en `config.json`). Las de recetas anteriores se listan pero no se comparan.
- **Detectores usados como clasificadores**: YOLOv8n y YOLO11n. El puntaje de cada imagen es la máxima confianza
  de sus cajas, con umbral 0.25.
- **Referencia**: el clasificador trivial que siempre dice "con paneles".

**Cómo leer los números:**

1. **Val también eligió los checkpoints**, así que todas las métricas son optimistas. Decirlo en el informe.
2. **Cada métrica va con un intervalo de 95 %** por bootstrap (se remuestrean las imágenes de val). Con 20
   imágenes sin paneles en val y 23 de Jamaica, una sola imagen mueve mucho las métricas.
3. **Una diferencia entre dos corridas es clara solo si su intervalo no incluye el 0.** Las diferencias se miden
   pareadas: las dos corridas sobre las mismas imágenes remuestreadas.
4. **Para comparar CNN contra detector, mirar sobre todo el AUC**: no depende del umbral, y el puntaje de YOLO no
   es una probabilidad calibrada.

Correr en Colab (no hace falta GPU) con *Ejecutar todas*. Se puede volver a correr cada vez que haya corridas
nuevas.

In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdi_tp import comparacion, entorno, resultados

rutas = entorno.preparar(solo_lectura=True)  # monta Drive, pero no deja crear ni guardar corridas

## 1. Qué corridas entran

`vigente` indica si la corrida entra en la comparación. Si falta alguna de las esperadas, todavía no se corrió (o
quedó incompleta) con la receta actual: la comparación se hace igual con las que hay.

In [ ]:
ESPERADAS = [f"{m}/{modo}" for m in ("mobilenetv2", "resnet50", "efficientnetb0")
             for modo in ("imagenet_congelado", "imagenet_finetuning", "aleatorio")]
ESPERADAS += [f"{m}/{modo}" for m in ("yolov8n", "yolo11n") for modo in ("coco_completo", "coco_backbone_congelado")]

corridas = comparacion.corridas_vigentes(resultados.cargar_corridas(rutas))
ELEGIDAS = corridas.loc[corridas["vigente"], "corrida"].tolist()
FAMILIA = dict(zip(corridas["corrida"], corridas["familia"]))

print(f"Receta vigente de las CNN: {comparacion.RECETA_VIGENTE}")
print("Entran:", len(ELEGIDAS))
faltan = [c for c in ESPERADAS if c not in ELEGIDAS]
print("Faltan (no corridas o con otra receta):", faltan or "ninguna")
viejas = corridas.loc[~corridas["vigente"], "corrida"].tolist()
print("Corridas con otra receta, que no se comparan:", viejas or "ninguna")
corridas[["corrida", "familia", "receta", "vigente", "umbral", "fecha"]]

## 2. Val: las 162 imágenes

Ordenadas por **F1 macro** (el criterio de los notebooks de clasificación): le da el mismo peso a la clase *sin
paneles*, así que el clasificador trivial queda abajo. Entre corchetes, el intervalo de 95 %.

In [ ]:
COLORES = {"clasificador": "#2a78d6", "detector": "#d6782a", "referencia": "#9a9a94"}
TITULOS = {"f1_macro": "F1 macro", "balanced_accuracy": "Balanced accuracy",
           "recall_sin_paneles": "Recall sin paneles", "roc_auc": "AUC"}


def con_intervalo(tabla):
    """Tabla legible: cada métrica como 'valor [inf, sup]'."""
    legible = tabla[["corrida"]].copy()
    legible.insert(1, "familia", legible["corrida"].map(FAMILIA))
    for metrica, titulo in TITULOS.items():
        legible[titulo] = [("—" if pd.isna(v) else f"{v:.3f} [{i:.3f}, {s:.3f}]")
                           for v, i, s in zip(tabla[metrica], tabla[f"{metrica}_inf"], tabla[f"{metrica}_sup"])]
    return legible.set_index("corrida")


def graficar(tabla, titulo):
    orden = tabla.iloc[::-1]  # el mejor arriba
    fig, axs = plt.subplots(1, len(TITULOS), figsize=(17, 0.42 * len(orden) + 1.6), sharey=True)
    colores = [COLORES[FAMILIA[c]] for c in orden["corrida"]]
    for ax, (metrica, nombre) in zip(axs, TITULOS.items()):
        valor = orden[metrica].to_numpy(float)
        error = np.vstack([valor - orden[f"{metrica}_inf"], orden[f"{metrica}_sup"] - valor])
        ax.barh(range(len(orden)), np.nan_to_num(valor), color=colores, height=0.6)
        ax.errorbar(np.nan_to_num(valor), range(len(orden)), xerr=np.nan_to_num(error), fmt="none",
                    ecolor="#333333", elinewidth=1, capsize=2)
        ax.set_xlim(0, 1.05)
        ax.set_title(nombre, loc="left", fontsize=11)
        ax.grid(axis="x", color="#e5e4df", lw=0.6)
        ax.set_axisbelow(True)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
    axs[0].set_yticks(range(len(orden)), orden["corrida"])
    fig.suptitle(titulo + "   (azul: clasificador · naranja: detector · gris: referencia; barras: intervalo de 95 %)",
                 x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    plt.show()


val = comparacion.intervalos(resultados.cargar_predicciones(rutas), ELEGIDAS).sort_values(
    ["f1_macro", "roc_auc"], ascending=False, ignore_index=True)
print(f"Val: {val['n'].iloc[0]} imágenes, {val['n_sin_paneles'].iloc[0]} sin paneles. "
      "Val también eligió los checkpoints: números optimistas.")
graficar(val, "Val")
con_intervalo(val)

## 3. Jamaica: las 23 imágenes de Kingston

Es lo más parecido al test (Kingston, con mayoría de imágenes sin paneles). Ordenadas por **balanced accuracy**,
que no depende de la proporción de clases (ver `docs/evaluacion_jamaica.md`). Con 23 imágenes los intervalos son
anchos: sirve para ver problemas grandes, no para elegir entre modelos parecidos.

In [ ]:
jamaica = comparacion.intervalos(resultados.cargar_predicciones(rutas), ELEGIDAS, solo_jamaica=True).sort_values(
    ["balanced_accuracy", "roc_auc"], ascending=False, ignore_index=True)
print(f"Jamaica: {jamaica['n'].iloc[0]} imágenes, {jamaica['n_sin_paneles'].iloc[0]} sin paneles.")
graficar(jamaica, "Jamaica")
con_intervalo(jamaica)

## 4. ¿Qué diferencias son claras?

Cada corrida contra la mejor: en val por F1 macro, y en Jamaica por balanced accuracy. `diferencia` es *corrida −
mejor* (negativa = peor). **`clara = True` solo si el intervalo no incluye el 0**: las demás corridas no se pueden
distinguir de la mejor con estos datos.

In [ ]:
predicciones = resultados.cargar_predicciones(rutas)
sin_referencia = [c for c in ELEGIDAS if FAMILIA[c] != "referencia"]
mejor_val = val.loc[val["corrida"].isin(sin_referencia), "corrida"].iloc[0]
mejor_jamaica = jamaica.loc[jamaica["corrida"].isin(sin_referencia), "corrida"].iloc[0]
print("Mejor en val (F1 macro):", mejor_val)
display(comparacion.diferencias(predicciones, ELEGIDAS, mejor_val, metrica="f1_macro").round(3))
print("Mejor en Jamaica (balanced accuracy):", mejor_jamaica)
comparacion.diferencias(predicciones, ELEGIDAS, mejor_jamaica, metrica="balanced_accuracy",
                        solo_jamaica=True).round(3)

## 5. Clasificador contra detector usado como clasificador

La pregunta de la consigna: ¿conviene un clasificador o un detector para decir si una imagen tiene paneles? Se
toma el mejor de cada familia en val (por F1 macro) y se comparan pareados, con **AUC** (no depende del umbral) y
con **F1 macro** (cada uno con su umbral: 0.5 la CNN, 0.25 el YOLO).

In [ ]:
mejores = {f: val.loc[val["corrida"].map(FAMILIA) == f, "corrida"].iloc[0]
           for f in ("clasificador", "detector") if (val["corrida"].map(FAMILIA) == f).any()}
print("Mejor de cada familia en val:", mejores)
if len(mejores) < 2:
    print("Falta al menos una corrida de cada familia para compararlas.")
else:
    filas = []
    for conjunto, solo_jamaica in (("val", False), ("jamaica", True)):
        for metrica in ("roc_auc", "f1_macro", "balanced_accuracy"):
            d = comparacion.diferencias(predicciones, [mejores["detector"]], mejores["clasificador"],
                                        metrica=metrica, solo_jamaica=solo_jamaica).iloc[0]
            filas.append({"conjunto": conjunto, "metrica": metrica, "detector − clasificador": d["diferencia"],
                          "inf": d["inf"], "sup": d["sup"], "clara": d["clara"]})
    display(pd.DataFrame(filas).round(3))

## 6. Detección: mAP de los detectores

Métricas de cajas que calcula Ultralytics, sobre val y sobre Jamaica (las guarda cada notebook de detección en
`metricas.json`). No tienen intervalo: son de una sola evaluación.

In [ ]:
detectores = [c for c in ELEGIDAS if FAMILIA[c] == "detector"]
deteccion = comparacion.tabla_deteccion(rutas.resultados_equipo, detectores)
if deteccion.empty:
    print("Todavía no hay detectores con métricas de detección guardadas.")
else:
    display(deteccion.pivot(index="corrida", columns="conjunto").round(3))

## Para el informe

- Decir que **val eligió los checkpoints y también se usó para medir**: los números son optimistas.
- Reportar cada métrica **con su intervalo**, y afirmar que un modelo es mejor solo cuando la diferencia es clara
  (sección 4).
- Comparar CNN contra detector **sobre todo con AUC**, y aclarar que el puntaje de YOLO es la máxima confianza de
  sus cajas, no una probabilidad.
- Jamaica tiene 23 imágenes: indica la tendencia para el test de Kingston, no permite elegir entre modelos
  parecidos.
- Todas las CNN usan la misma receta (sección *Receta común de clasificación* del README) y una sola semilla:
  la variabilidad entre semillas no está medida.